# Get price history (500 candles)

For a future analysis

In [2]:
import requests
import pandas as pd
import logging
import time
import datetime
import os
import csv
import asyncio

# from dotenv import load_dotenv  # we cannot use it cause it's not in Anaconda Base list 

# Our own classes to gather some data 
from trading_data_classes import GetDataTradingView, DataWorks

tv = GetDataTradingView()
dw = DataWorks()

In [ ]:
def make_a_record_from_tv(symbol, exchange, interval, n_bars, file_path):
    df = tv.get_hist(           
        symbol = symbol,        #  Instrument name, format like "BTCUSDT"
        exchange = exchange,    #  Exchange, source of the quotes (from which TradingView get quotes)
                                #               format "BINANCE"
        interval = interval,    #  str value like "5" --> means 5 minutes
        n_bars = n_bars,        #  How many bars (candles) we're requesting: 
                                #               1 --> only the last one, up to 10_000 --> for history (paywall after ~10k)
    )
    # file_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), file_path)
    try: 
        timestamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
        write_header = False
        if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
            write_header = True
        with open(file_path, 'a', encoding='utf-8') as f:
            if write_header:
                f.write("instrument,timestamp_utc,open_price,high_price,low_price,close_price,record_timestamp_utc\n")
            # Write all rows from the DataFrame
            for idx, row in df.iterrows():
                """
                Writing the row with the following data: 
                
                (0) exchange:instument (symbol)
                (1) candle datettime (pandas to_datetime) with the format 2025-09-13 17:30:00115905.88
                (2) open_price
                (3) high_price
                (4) low_price
                (5) close_price
                (6) record_timestamp (when the record has been put into the file)
                
                """
                f.write(f'{row.iloc[0]},'
                        f'{pd.to_datetime(idx)},'
                        f'{row.iloc[1]},{row.iloc[2]},{row.iloc[3]},{row.iloc[4]},'
                        f'{timestamp}\n')
        dw.write_log_line(text = f"Candle of '{row.iloc[0]}' has written to {file_path}"
                                    f" with the time {pd.to_datetime(idx)}")
    except Exception as e:
        logging.error(f"Error writing to log file: {e}")
        print(f"Error writing to log file: {e}")
    return

In [ ]:
# Get 500 rows of every instrument (5 minutes candles)

make_a_record_from_tv(symbol = "MAG7",                                  
                      exchange = "LSE", 
                      interval = "5", 
                      n_bars = 500, 
                      file_path = 'data/history/mag7.csv')

make_a_record_from_tv(symbol = "BTCUSDT",                                  
                        exchange = "BINANCE", 
                        interval = "5", 
                        n_bars = 500, 
                        file_path = 'data/history/btcusdt.csv'), 

make_a_record_from_tv(symbol = "TONUSDT",                                 
                        exchange = "BINANCE", 
                        interval = "5", 
                        n_bars = 500, 
                        file_path = 'data/history/tonusdt.csv'),

Candle of 'BINANCE:BTCUSDT' has written to data/history/btcusdt.csv with the time 2025-09-20 17:50:00+00:00
Candle of 'BINANCE:TONUSDT' has written to data/history/tonusdt.csv with the time 2025-09-20 17:50:00+00:00


(None,)